# GenAI Task 3 — AI Resume Screening System with LangSmith Tracing
**Internship:** Data Science Internship – February 2026 | Innomatics Research Labs  
**Tools:** Python · LangChain · LangSmith · OpenAI GPT-3.5

---

## What This System Does
Takes a resume + job description as input and runs a 4-step AI pipeline:

```
Resume + Job Description
        ↓
  Step 1: Skill Extraction
  (Extract skills, experience, tools from resume)
        ↓
  Step 2: Matching Logic
  (Compare extracted skills against JD requirements)
        ↓
  Step 3: Scoring (0–100)
  (Quantify how well the candidate fits)
        ↓
  Step 4: Explanation
  (Natural language reasoning for the score)
        ↓
  Final Output: Score + Match Report + Explanation
        ↓
  LangSmith Traces all 4 steps for debugging
```

**Three candidate profiles tested:** Strong | Average | Weak

---
## Cell 1 — Install Dependencies

In [ ]:
# Core packages needed for this project
# langchain-core    : LCEL, PromptTemplate, output parsers
# langchain-openai  : OpenAI LLM integration
# langsmith         : Tracing and monitoring of LLM pipelines
!pip install langchain langchain-core langchain-openai langsmith openai --quiet

print("All packages installed successfully.")

---
## Cell 2 — API Keys and LangSmith Tracing Setup

### How to get your LangSmith API key (free):
1. Go to [smith.langchain.com](https://smith.langchain.com)
2. Sign up with Google or email (free tier is enough)
3. Go to **Settings → API Keys → Create API Key**
4. Copy and paste it below

> Setting `LANGCHAIN_TRACING_V2=true` automatically sends every `.invoke()` call to LangSmith for tracing.

In [ ]:
import os

# ── OpenAI API Key ────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# ── LangSmith Tracing Configuration ──────────────────────────────
# These 4 env variables enable automatic tracing of every LangChain call
os.environ["LANGCHAIN_TRACING_V2"]  = "true"           # Enable tracing
os.environ["LANGCHAIN_API_KEY"]     = "your-langsmith-api-key-here"  # LangSmith key
os.environ["LANGCHAIN_PROJECT"]     = "Resume-Screening-System"       # Project name in LangSmith
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"

print("Environment configured.")
print(f"LangSmith Tracing : {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"LangSmith Project : {os.environ.get('LANGCHAIN_PROJECT')}")
print("\nNote: Replace placeholder keys with your actual API keys before running.")

---
## Cell 3 — Import Libraries

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable   # Decorator to tag functions for LangSmith tracing
import json
import re

# Initialize the LLM — temperature=0 for consistent, deterministic evaluation
# We don't want creative variation when scoring resumes
screening_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Reusable output parser — strips formatting from LLM response
text_parser = StrOutputParser()

print("Libraries imported.")
print(f"Model: {screening_llm.model_name}")

---
## Cell 4 — Job Description and 3 Candidate Resumes

Three profiles designed to produce clearly different scores:
- **Candidate A (Strong):** Matches most JD requirements
- **Candidate B (Average):** Partial match — some relevant skills, gaps in others  
- **Candidate C (Weak):** Minimal overlap with the job requirements

In [ ]:
# ── Job Description ───────────────────────────────────────────────
JOB_DESCRIPTION = """
Role: Data Scientist
Company: TechCorp Analytics

Required Skills:
- Python (pandas, numpy, scikit-learn)
- Machine Learning (classification, regression, clustering)
- Deep Learning (TensorFlow or PyTorch)
- SQL and database querying
- Data visualization (Matplotlib, Seaborn, Power BI or Tableau)
- Statistical analysis and hypothesis testing
- Experience with NLP or Computer Vision is a plus

Required Experience:
- Minimum 2 years of hands-on data science or ML experience
- Experience deploying ML models to production
- Exposure to cloud platforms (AWS, GCP, or Azure)

Nice to Have:
- LangChain or GenAI experience
- MLflow or experiment tracking tools
- Familiarity with Docker and CI/CD pipelines
"""

# ── Candidate A: Strong Profile ───────────────────────────────────
RESUME_STRONG = """
Name: Arjun Mehta
Experience: 4 years in Data Science and Machine Learning

Technical Skills:
- Python: pandas, numpy, scikit-learn, matplotlib, seaborn
- Machine Learning: classification, regression, clustering, ensemble methods
- Deep Learning: TensorFlow, Keras, PyTorch — built and deployed CNN and LSTM models
- NLP: BERT fine-tuning, text classification, named entity recognition
- SQL: complex queries, joins, window functions — PostgreSQL and MySQL
- Visualization: Power BI dashboards, Tableau, Matplotlib
- Cloud: AWS (S3, EC2, SageMaker), model deployment on AWS Lambda
- Tools: MLflow experiment tracking, Docker, Git, Jupyter
- GenAI: LangChain, OpenAI API, RAG pipelines

Experience:
- Data Scientist at DataVision Pvt Ltd (2021–2025)
  - Built customer churn prediction model (91% accuracy, deployed to production)
  - Developed NLP pipeline for sentiment analysis on 2M+ product reviews
  - Led A/B testing framework used across 3 product teams
- ML Intern at StartupX (2020–2021)
  - Built recommendation engine using collaborative filtering

Education: B.E. Computer Science, NIT Surathkal, 2020
"""

# ── Candidate B: Average Profile ─────────────────────────────────
RESUME_AVERAGE = """
Name: Priya Sharma
Experience: 1.5 years in data analytics and some ML work

Technical Skills:
- Python: pandas, numpy, basic matplotlib
- Machine Learning: linear regression, logistic regression — mostly from Coursera projects
- SQL: basic SELECT queries, joins — MySQL
- Visualization: Excel charts, some Matplotlib
- Tools: Jupyter Notebook, Git

Experience:
- Data Analyst at RetailCo (2023–2025)
  - Built sales reports and dashboards using Excel and Python
  - Wrote SQL queries to pull data from internal databases
  - Ran a logistic regression model to predict late deliveries (personal project)
- Intern at FinanceApp (2022–2023)
  - Data entry, basic data cleaning in Python

Education: B.Sc. Statistics, Bangalore University, 2022

Note: No deep learning experience. No cloud platform exposure. No production deployments.
"""

# ── Candidate C: Weak Profile ─────────────────────────────────────
RESUME_WEAK = """
Name: Rohit Verma
Experience: 6 months, mostly web development

Technical Skills:
- HTML, CSS, JavaScript
- React.js, Node.js
- Basic Python — learned from YouTube tutorials
- Some Excel usage

Experience:
- Junior Web Developer at WebStudio (2024–2025)
  - Built frontend components using React
  - Basic REST API integration with Node.js
- College Projects:
  - Library management system in Java
  - Student portal website in HTML/CSS

Education: B.E. Information Science, Visvesvaraya University, 2024

Note: No machine learning experience. No SQL. No data science background.
"""

# Package candidates for easy iteration later
candidates = {
    "Candidate A (Strong)" : RESUME_STRONG,
    "Candidate B (Average)": RESUME_AVERAGE,
    "Candidate C (Weak)"   : RESUME_WEAK
}

print(f"Job Description loaded.")
print(f"Candidates loaded   : {list(candidates.keys())}")

---
## Cell 5 — Prompt Templates (modular/prompts/)

Four separate prompt templates — one per pipeline step.  
Key rule embedded in all prompts: **Do NOT assume skills not present in the resume.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PROMPT 1: SKILL EXTRACTION
#  Extracts structured information from raw resume text
# ═══════════════════════════════════════════════════════════════
extraction_prompt = PromptTemplate(
    input_variables=["resume_text"],
    template="""
You are a precise resume parser. Your job is to extract structured information from the resume below.

STRICT RULES:
- Only extract skills, tools, and experience that are EXPLICITLY mentioned in the resume.
- Do NOT infer, assume, or add skills that are not written in the text.
- If a field has no information, write "Not mentioned".

Resume:
{resume_text}

Extract and list the following:

1. TECHNICAL SKILLS:
   (List every programming language, framework, library, and technical tool mentioned)

2. MACHINE LEARNING / AI EXPERIENCE:
   (List ML algorithms, models, or AI-related work mentioned)

3. TOTAL YEARS OF EXPERIENCE:
   (Based only on what is written — do not estimate)

4. TOOLS AND PLATFORMS:
   (List databases, cloud platforms, DevOps tools, visualization tools)

5. PRODUCTION DEPLOYMENT EXPERIENCE:
   (Has the candidate deployed ML models? Yes/No — based only on resume text)

Output as a clean structured list. No extra commentary.
"""
)

# ═══════════════════════════════════════════════════════════════
#  PROMPT 2: MATCHING LOGIC
#  Compares extracted profile against JD requirements
# ═══════════════════════════════════════════════════════════════
matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a technical recruiter. Compare the candidate's profile against the job requirements.

STRICT RULES:
- Only use information from the extracted profile below. Do NOT assume any skills.
- For each requirement, clearly state: MATCHED / PARTIAL / NOT MATCHED.
- Be specific — name the exact skill or evidence from the profile.

Candidate Profile (extracted from resume):
{extracted_profile}

Job Description Requirements:
{job_description}

Produce a match report with these sections:

MATCHED REQUIREMENTS:
(List JD requirements that the candidate clearly meets)

PARTIAL MATCHES:
(List requirements where there is some overlap but not full coverage)

MISSING REQUIREMENTS:
(List JD requirements with no evidence in the candidate profile)

BONUS POINTS:
(Any nice-to-have skills from the JD that the candidate has)
"""
)

# ═══════════════════════════════════════════════════════════════
#  PROMPT 3: SCORING
#  Assigns a numeric fit score based on the match report
# ═══════════════════════════════════════════════════════════════
scoring_prompt = PromptTemplate(
    input_variables=["match_report", "job_description"],
    template="""
You are a hiring evaluation system. Based on the match report below, assign a numeric fit score.

SCORING RULES:
- Score range: 0 to 100
- Base the score ONLY on evidence in the match report
- Use this scoring guide:
  * 80–100: Strong fit — meets most/all required skills + experience
  * 50–79 : Moderate fit — meets some requirements, clear gaps
  * 20–49 : Weak fit — limited overlap, major gaps
  * 0–19  : Poor fit — minimal or no relevant skills

Scoring Weights:
- Core technical skills match: 40 points
- Years of experience match: 20 points
- Production deployment experience: 15 points
- Cloud/tools exposure: 15 points
- Bonus/nice-to-have skills: 10 points

Match Report:
{match_report}

Job Description (for reference):
{job_description}

Output format:
FINAL SCORE: [number]/100

SCORE BREAKDOWN:
- Core Technical Skills: [X]/40
- Years of Experience: [X]/20
- Production Deployment: [X]/15
- Cloud/Tools: [X]/15
- Bonus Skills: [X]/10
"""
)

# ═══════════════════════════════════════════════════════════════
#  PROMPT 4: EXPLANATION
#  Generates human-readable hiring recommendation
# ═══════════════════════════════════════════════════════════════
explanation_prompt = PromptTemplate(
    input_variables=["score_result", "match_report", "candidate_name"],
    template="""
You are a senior hiring manager writing a candidate evaluation summary.

RULES:
- Be honest and specific — no vague praise
- Base all statements on the match report and score only
- Do NOT add skills or experience not present in the data
- Keep the tone professional but direct

Candidate: {candidate_name}
Score Result: {score_result}
Match Report: {match_report}

Write a structured evaluation with:

RECOMMENDATION: (Shortlist / Hold / Reject)

STRENGTHS:
(2–3 specific things this candidate does well for this role)

GAPS:
(2–3 specific missing requirements that affected the score)

HIRING MANAGER SUMMARY:
(2–3 sentences explaining the recommendation in plain English)
"""
)

print("All 4 prompt templates defined.")
print("  - extraction_prompt  : Skill Extraction")
print("  - matching_prompt    : Requirement Matching")
print("  - scoring_prompt     : Score Assignment")
print("  - explanation_prompt : Hiring Explanation")

---
## Cell 6 — LangChain Chains (LCEL)

Each chain = `prompt | llm | parser`  
All 4 chains use the pipe operator (`|`) — this is LCEL (LangChain Expression Language).

In [ ]:
# ── Build 4 independent chains using LCEL pipe syntax ─────────────
# Each chain: PromptTemplate → LLM → StrOutputParser

# Chain 1: Extracts structured skills/experience from resume text
extraction_chain = extraction_prompt | screening_llm | text_parser

# Chain 2: Compares extracted profile to JD, produces match report
matching_chain = matching_prompt | screening_llm | text_parser

# Chain 3: Assigns numeric score based on match report
scoring_chain = scoring_prompt | screening_llm | text_parser

# Chain 4: Generates hiring recommendation + explanation
explanation_chain = explanation_prompt | screening_llm | text_parser

print("All 4 LCEL chains built:")
print("  extraction_chain  → prompt | llm | parser")
print("  matching_chain    → prompt | llm | parser")
print("  scoring_chain     → prompt | llm | parser")
print("  explanation_chain → prompt | llm | parser")

---
## Cell 7 — Main Screening Pipeline Function

The `@traceable` decorator on this function tells LangSmith to record this entire run as one trace,  
with all 4 chain calls visible as sub-steps inside it.

In [ ]:
@traceable(name="Resume-Screening-Pipeline")  # LangSmith traces this function
def run_screening_pipeline(candidate_label, resume_text, jd_text):
    """
    Full 4-step AI resume screening pipeline.

    Steps:
        1. Extract skills/experience from resume
        2. Match extracted profile against JD requirements
        3. Assign a numeric fit score (0–100)
        4. Generate explanation and hiring recommendation

    Args:
        candidate_label (str): Display name for the candidate
        resume_text (str)    : Raw resume content
        jd_text (str)        : Job description text

    Returns:
        dict: Results from all 4 pipeline steps
    """

    separator = "=" * 65
    print(f"\n{separator}")
    print(f"  SCREENING: {candidate_label}")
    print(separator)

    # ── STEP 1: Skill Extraction ──────────────────────────────────
    print("\n[Step 1] Extracting skills and experience...")
    extracted_profile = extraction_chain.invoke({"resume_text": resume_text})
    print(extracted_profile)

    # ── STEP 2: Matching Logic ────────────────────────────────────
    print("\n" + "-" * 65)
    print("[Step 2] Matching profile against job requirements...")
    match_report = matching_chain.invoke({
        "extracted_profile": extracted_profile,
        "job_description"  : jd_text
    })
    print(match_report)

    # ── STEP 3: Scoring ───────────────────────────────────────────
    print("\n" + "-" * 65)
    print("[Step 3] Calculating fit score...")
    score_result = scoring_chain.invoke({
        "match_report"   : match_report,
        "job_description": jd_text
    })
    print(score_result)

    # ── STEP 4: Explanation ───────────────────────────────────────
    print("\n" + "-" * 65)
    print("[Step 4] Generating hiring recommendation...")
    explanation = explanation_chain.invoke({
        "score_result"   : score_result,
        "match_report"   : match_report,
        "candidate_name" : candidate_label
    })
    print(explanation)

    print(f"\n{separator}")
    print(f"  Pipeline complete for {candidate_label}")
    print(separator)

    # Return all results for the summary report
    return {
        "candidate"        : candidate_label,
        "extracted_profile": extracted_profile,
        "match_report"     : match_report,
        "score_result"     : score_result,
        "explanation"      : explanation
    }


print("run_screening_pipeline() defined and ready.")
print("LangSmith @traceable decorator is active — all runs will be traced.")

---
## Cell 8 — Run Pipeline on All 3 Candidates

Each run creates a separate trace in LangSmith.  
You should see 3 traces in your LangSmith project after this cell completes.

In [ ]:
# Store all results for the comparison table at the end
all_results = []

# Run the full pipeline for each candidate
# Each call = 1 LangSmith trace with 4 sub-steps
for candidate_label, resume_text in candidates.items():
    result = run_screening_pipeline(
        candidate_label=candidate_label,
        resume_text=resume_text,
        jd_text=JOB_DESCRIPTION
    )
    all_results.append(result)

print("\nAll 3 candidates screened.")
print("Check your LangSmith dashboard for the 3 traces:")
print("  https://smith.langchain.com → Project: Resume-Screening-System")

---
## Cell 9 — Comparison Summary Table

In [ ]:
def extract_score(score_text):
    """
    Pull the numeric score from the scoring chain output.
    Looks for pattern 'FINAL SCORE: XX/100'
    """
    match = re.search(r'FINAL SCORE:\s*(\d+)/100', score_text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return "N/A"


def extract_recommendation(explanation_text):
    """
    Pull the RECOMMENDATION line from the explanation output.
    """
    match = re.search(r'RECOMMENDATION:\s*(.*)', explanation_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return "N/A"


# Build summary
print("=" * 65)
print("       RESUME SCREENING — FINAL COMPARISON SUMMARY")
print("=" * 65)
print(f"{'Candidate':<30} {'Score':>8}   {'Recommendation'}")
print("-" * 65)

for res in all_results:
    score = extract_score(res["score_result"])
    recommendation = extract_recommendation(res["explanation"])
    print(f"{res['candidate']:<30} {str(score)+'/100':>8}   {recommendation}")

print("=" * 65)
print("\nFull trace details available at: https://smith.langchain.com")

---
## Cell 10 — LangSmith Tracing: What to Screenshot

After running Cell 8, go to [smith.langchain.com](https://smith.langchain.com) and take screenshots of:

1. **Project Overview** — Shows 3 runs (Strong, Average, Weak) listed
2. **Individual Trace** — Click on one run → shows all 4 pipeline steps inside it
3. **Debug Example** — If any step produced unexpected output, the trace shows exactly which step failed

These screenshots are your proof of tracing for the submission.

In [ ]:
# ── LangSmith Tracing Verification ───────────────────────────────
# This cell shows what tracing information was captured

print("LangSmith Tracing Summary")
print("=" * 55)
print(f"Project Name  : {os.environ.get('LANGCHAIN_PROJECT')}")
print(f"Tracing Status: {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"Total Runs    : {len(all_results)}")
print()
print("Each trace contains 4 sub-steps:")
print("  [1] extraction_chain  — Skill Extraction")
print("  [2] matching_chain    — Requirement Matching")
print("  [3] scoring_chain     — Score Calculation")
print("  [4] explanation_chain — Hiring Explanation")
print()
print("Dashboard URL: https://smith.langchain.com")
print("Navigate to: Projects → Resume-Screening-System")
print()
print("What you will see in LangSmith:")
print("  - Each candidate = 1 parent trace")
print("  - Inside each trace = 4 LLM calls with inputs/outputs")
print("  - Latency, token usage, and errors shown per step")
print("  - Use this to debug: which step gave a wrong output?")

---
## Cell 11 — Bonus: Structured JSON Output

Re-runs the pipeline for one candidate with JSON-structured output.  
Makes the output machine-parseable — useful for real production systems.

In [ ]:
# Bonus: JSON-structured scoring output
json_scoring_prompt = PromptTemplate(
    input_variables=["match_report", "candidate_name"],
    template="""
You are an automated hiring evaluation API. Output ONLY valid JSON. No extra text before or after.

Based on the match report, produce a structured evaluation.

Candidate: {candidate_name}
Match Report: {match_report}

Return this exact JSON structure (fill in the values):
{{
  "candidate_name": "string",
  "final_score": integer between 0 and 100,
  "recommendation": "Shortlist" or "Hold" or "Reject",
  "score_breakdown": {{
    "core_technical_skills": integer out of 40,
    "years_of_experience": integer out of 20,
    "production_deployment": integer out of 15,
    "cloud_and_tools": integer out of 15,
    "bonus_skills": integer out of 10
  }},
  "top_strengths": ["strength 1", "strength 2"],
  "key_gaps": ["gap 1", "gap 2"]
}}

RULES:
- Output ONLY the JSON object above. Nothing else.
- Do NOT include markdown backticks or labels.
- All values must be based only on the match report provided.
"""
)

# Build JSON chain
json_chain = json_scoring_prompt | screening_llm | text_parser

# Run on all 3 candidates
print("=" * 55)
print("BONUS: JSON-Structured Output for All Candidates")
print("=" * 55)

for res in all_results:
    print(f"\nCandidate: {res['candidate']}")
    print("-" * 55)

    json_raw = json_chain.invoke({
        "match_report"   : res["match_report"],
        "candidate_name" : res["candidate"]
    })

    # Parse and pretty-print the JSON
    try:
        # Clean any accidental markdown backticks
        clean_json = json_raw.strip().replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean_json)
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print("[Raw output — JSON parsing failed]")
        print(json_raw)

---
## Summary

| Component | Implementation |
|---|---|
| LLM | `ChatOpenAI` (gpt-3.5-turbo, temperature=0) |
| Prompt Templates | 4 separate `PromptTemplate` objects (one per step) |
| Chains | LCEL pipe syntax: `prompt \| llm \| parser` |
| Tracing | LangSmith `@traceable` decorator + env variables |
| Candidates tested | Strong, Average, Weak (3 runs → 3 LangSmith traces) |
| Anti-hallucination | Explicit rule: "Do NOT assume skills not in resume" |
| Bonus | JSON-structured output with all 3 candidates |

### Pipeline Architecture
```
Resume + JD
    ↓
extraction_chain   → Structured skills/experience list
    ↓
matching_chain     → Matched / Partial / Missing requirements
    ↓
scoring_chain      → Numeric score (0–100) with breakdown
    ↓
explanation_chain  → Recommendation + Strengths + Gaps
    ↓
LangSmith          → Full trace of all 4 steps per candidate
```

---
*Task 3 — GenAI Internship | Innomatics Research Labs | February 2026*  
*Portfolio: siddharthkaulagi.vercel.app*